# Structured streaming Demo

In [1]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




In [2]:
# initialize the spark session
import pyspark

# the course hardcodes the connector 4.1.0, but this image ships spark 4.2.0 and the two are
# not binary compatible (NoSuchMethodError on HDFSMetadataLog at the first batch). The kafka
# connector must match the spark version AND the scala version (_2.13 here), so we build the
# coordinate from the running spark instead of writing it down.
KAFKA_PKG = f"org.apache.spark:spark-sql-kafka-0-10_2.13:{pyspark.__version__}"
print("spark", pyspark.__version__, "| connector:", KAFKA_PKG)

spark = SparkSession.builder     .appName("KafkaStreamingPrototype")     .config("spark.jars.packages", KAFKA_PKG)     .getOrCreate()

spark 4.2.0 | connector: org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0


In [3]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [4]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [5]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [6]:

# Use console output (goes to Docker logs)
streaming_query_df = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

---

# Lab

Everything above is the demo of the course, unchanged. Below is step 12 of the lab: one exploration with a
tumblin window and one with an overlapping window.

The demo already uses a tumbling window of 1 hour (`window(event_time, "1 hour")`: when only one duration is
given, the windows are side by side and an edit belongs to exactly one of them). We redo it on 10 minutes so
the result moves while we watch the logs, then we do the same count with a window that slides.

## Tumbling window — edits per 10 minutes, bot vs human

One duration only, so the windows do not overlap: `10:00-10:10`, `10:10-10:20`, ... and each edit is counted
once.

In [7]:
# same aggregation as the demo, but on a 10 minute tumbling window
tumbling_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp")).cast("timestamp")) \
    .groupBy(
        window(col("event_time"), "10 minutes"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

tumbling_query = tumbling_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .queryName("tumbling_10min") \
    .start()

## Overlapping window — 10 minutes sliding every 5 minutes

Two durations: the window still lasts 10 minutes but a new one starts every 5 minutes, so the windows overlap
and **one edit is counted in two windows** (`10:00-10:10` and `10:05-10:15`). So the total on all the windows is two times the tumbling one, but a window alone is not doubled : the
windows that exist in the two are identical, it's the windows in the middle that add the duplicates. A
sliding window is made to smooth a curve, not to count without duplicates.

In [8]:
# window(column, windowDuration, slideDuration): the 2nd duration is what makes it overlap
overlapping_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp")).cast("timestamp")) \
    .groupBy(
        window(col("event_time"), "10 minutes", "5 minutes"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

overlapping_query = overlapping_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .queryName("sliding_10min_every_5min") \
    .start()

## Let the queries run and check them

The three queries run in the background threads and print their batches into the container logs
(`docker logs -f pyspark_notebook` on the host). We wait 2 minutes so a few micro-batches are
produced, then we print and will analyze the state of each query.

In [9]:
import time

# a streaming query runs in a background thread: start() returns at once and the batches are
# printed in the container logs. So we let them run a bit before looking at the result,
# otherwise a "run all" reaches the end of the notebook before the first batch is even built.
WATCH_SEC = 120
print(f"letting the queries run for {WATCH_SEC}s (output goes to: docker logs -f pyspark_notebook)")
time.sleep(WATCH_SEC)

for q in spark.streams.active:
    p = q.lastProgress
    print(f"{q.name:<26} active={q.isActive} batches={p['batchId'] + 1 if p else 0} "
          f"rows_in_last_batch={p['numInputRows'] if p else 0}")
    if q.exception():
        print("   ERROR:", str(q.exception())[:300])

letting the queries run for 120s (output goes to: docker logs -f pyspark_notebook)
tumbling_10min             active=True batches=2 rows_in_last_batch=0


TypeError: unsupported format string passed to NoneType.__format__

## Results

The three queries print in the logs of the container, one block `Batch: 0` each. We recognize them with the
windows : the demo has windows of 1 hour, the tumbling has only bounds on :00, :10, :20... and the
overlapping has also bounds on :05, :15, :45, :55.

### Demo of the course - tumbling window of 1 hour

![demo, 1 hour windows](screenshots/01_demo_tumbling_1hour.png)

### Our tumbling window of 10 minutes

![tumbling 10 minutes](screenshots/03_tumbling_10min.png)

**The tumbling is coherent with the demo.** On the hour 09:00-10:00 of september 23 the demo counts 482
humans and 114 bots in total, in the other hand our tumbling windows of this hour give 174 + 308 = 482 for the humans and 36 + 78 = 114
for the bots. So we obtain exactly the same result. It's normal, a tumbling window cuts the time without hole and without
overlapping, so the small windows are just the big one splitted. It's just a copy paste. 

### Our overlapping window of 10 minutes sliding every 5 minutes

![overlapping 10 minutes every 5](screenshots/02_overlapping_10min_every_5min.png)

**The overlapping counts some edits two times.** The tumbling gives 09:50-10:00 = 308 and 10:00-10:10 = 298
for the humans. The overlapping gives the same two windows, but it adds 09:55-10:05 = 348 which doesn't
exist in the tumbling.It's because this one takes the end of the first and the beginning of the second. So the edits
between 09:55 and 10:00 are counted in two windows(duplicate).It's exactly what we've seen in the lecture which is called at most-once. Attention, it's the total on all the windows that is two
times bigger, one window alone is not doubled.

### What we see in the numbers

20% of the edits are made by bots(114 bots against 482 humans on 09:00-10:00). That's what the demo of
the course wanted to show us. 

Event time and not processing time. We are the 1st of october but the windows are the 23 and 24 of
september, because the producer of the course has `since='20260209'` and replays the history again and again. And the two
days are mixed in the same batch (09-23 09:00, then 09-24 07:00, then 09-23 10:00...). Spark doesn't care
because it groups on the `event_time` (the field `timestamp` of the message) and not on the order of
arrival. That's not streaming like in kafka. It's exactly the event time vs processing time of the course.

Only one batch.(Batch 0 each time)The three queries read with `startingOffsets: earliest` so the first micro-batch eats
all the topic at once. There is no `Batch: 1` after because our producer is finished (it runs only 10
minutes) so nothing new can arrives to have a batch 1. If we restart it the same windows are updated and the counts grow, it's the
`complete` output mode.

## Stop the queries

Commented out on purpose, see the comment in the cell.

In [11]:
# a streaming query never stops by itself, we have to ask. Kept commented out on purpose:
# a "run all" would otherwise stop the queries one second after starting them, and no batch
# would ever be printed. Run this cell by hand when you are done watching the logs.

# for q in spark.streams.active:
#     q.stop()
# print("stopped,", len(spark.streams.active), "queries left")